# Game Intelligence AI — Stage 6: Dataset Curation & Quality Control

This notebook triages the Stage 5 audit into a human-review queue. It does **not** download source files, extract code, create training examples, approve licenses, or train a model.

**Critical:** repository license labels and nearby LICENSE/NOTICE paths are evidence only, not legal conclusions. Every candidate remains unapproved until file-level and intended-use review. No Google Drive is used; Colab `/content` is temporary.

In [ ]:
from pathlib import Path
import json, csv, datetime, collections
WORK=Path('/content/game_intelligence_stage6')
INPUT=WORK/'input'; OUTPUT=WORK/'output'
INPUT.mkdir(parents=True,exist_ok=True); OUTPUT.mkdir(parents=True,exist_ok=True)
print('Workspace ready:',WORK)
print('Next: run the upload cell manually. It is intentionally not automatic.')

## 1. Upload the Stage 5 audit

Run this cell separately after setup. It is not part of an automatic Run all path, so Colab will not hang waiting for an upload.

In [ ]:
from google.colab import files
uploaded=files.upload()
names=[n for n in uploaded if n.lower().endswith('.json')]
if not names: raise ValueError('Upload stage5_repository_audit.json')
AUDIT_PATH=INPUT/'stage5_repository_audit.json'
AUDIT_PATH.write_bytes(uploaded[names[0]])
print('Loaded:',AUDIT_PATH,'bytes:',AUDIT_PATH.stat().st_size)

## 2. Parse the candidate inventory

The parser supports common Stage 5 JSON shapes and fails loudly if no candidate list is found.

In [ ]:
audit=json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
def locate(obj):
    if isinstance(obj,dict):
        for k in ('candidate_rows','candidates','candidate_inventory','code_candidates'):
            v=obj.get(k)
            if isinstance(v,list) and (not v or isinstance(v[0],dict)): return v,k
        for v in obj.values():
            got=locate(v)
            if got: return got
    elif isinstance(obj,list):
        for v in obj:
            got=locate(v)
            if got: return got
    return None
found=locate(audit)
if not found: raise ValueError('Candidate inventory not found. Inspect JSON structure; do not proceed with an empty dataset.')
candidates,key=found
print('Inventory key:',key,'| candidate rows:',len(candidates))
print('Top-level keys:',list(audit) if isinstance(audit,dict) else type(audit).__name__)

## 3. Conservative candidate triage

This is a transparent *review-priority filter*, not a license detector. Likely vendored, generated, bundled, media, legacy/WIP, minified, and oversized files are held out of the first review queue. Nothing is silently deleted.

In [ ]:
EXCLUDED={'node_modules','vendor','vendors','third_party','third-party','external','dist','build','out','coverage','.git','__pycache__','.venv','venv','addons','plugins','plugin','assets','asset','media','images','image','textures','fonts','audio','sounds','music','video','videos','models','3d','sprites','tilesets','export','exports','cache','tmp','temp'}
RISK={'_wip','wip','deprecated','legacy','old','archive','archived','backup'}
EXTS={'.gd','.gdshader','.cs','.cpp','.h','.hpp','.c','.js','.mjs','.ts','.tsx','.jsx','.py','.lua','.java','.kt','.swift','.rs','.json','.yaml','.yml','.toml','.xml','.html','.css','.glsl'}
MAX_BYTES=250_000
def triage(r):
    path=str(r.get('path','')).replace('\\','/')
    parts=[p.lower() for p in path.split('/') if p]
    name=parts[-1] if parts else ''
    reasons=[]
    if not path: reasons.append('missing_path')
    if any(p in EXCLUDED for p in parts[:-1]): reasons.append('likely_vendor_asset_or_generated_directory')
    if any(p in RISK or p.startswith('_wip') for p in parts): reasons.append('wip_legacy_or_archive_path')
    if __import__('pathlib').Path(name).suffix.lower() not in EXTS: reasons.append('extension_not_in_review_allowlist')
    if name.endswith(('.min.js','.min.css')) or '.bundle.' in name: reasons.append('minified_or_bundled_file')
    size=r.get('size_bytes')
    if isinstance(size,(int,float)) and size>MAX_BYTES: reasons.append('oversized_file')
    if isinstance(size,(int,float)) and size<=0: reasons.append('empty_or_invalid_size')
    if not all(r.get(k) for k in ('repo','commit_sha','blob_sha')): reasons.append('missing_source_identity_or_hash')
    return ('HOLD_EXCLUDED_FROM_FIRST_REVIEW',';'.join(reasons)) if reasons else ('HUMAN_REVIEW_REQUIRED','candidate_for_manual_file_and_license_review')
review=[]; held=[]; seen=set()
for source in candidates:
    if not isinstance(source,dict): continue
    r=dict(source); status,reason=triage(r)
    r.update(stage6_triage_status=status,stage6_triage_reason=reason,
             approved_for_training=False,license_decision='NOT_ASSESSED',
             technical_quality_decision='NOT_ASSESSED')
    ident=(r.get('repo'),r.get('commit_sha'),r.get('path'))
    if ident in seen:
        r['stage6_triage_status']='HOLD_DUPLICATE_IN_AUDIT'
        r['stage6_triage_reason']='duplicate_repo_commit_path_record'
    seen.add(ident)
    (review if r['stage6_triage_status']=='HUMAN_REVIEW_REQUIRED' else held).append(r)
print('Human review queue:',len(review),'| held/excluded:',len(held))

## 4. Add transparent topic hints

Hints are based only on path names and are not semantic validation. A human must inspect actual contents.

In [ ]:
def topic_hint(path):
    p=str(path).lower()
    for topic,terms in [
      ('physics',['physics','collision','rigid','raycast','movement']),
      ('npc_ai',['ai','npc','enemy','navigation','pathfinding','behavior']),
      ('level_design',['level','scene','map','tile','world']),
      ('tools_pipeline',['editor','tool','import','build','pipeline']),
      ('architecture',['manager','autoload','singleton','service','system'])]:
        if any(t in p for t in terms): return topic
    return 'needs_human_classification'
for r in review:
    r['suggested_topic_hint']=topic_hint(r.get('path',''))
    r['human_reviewer']=''
    r['review_notes']=''
    r['license_evidence_checked']='NO'
    r['file_header_checked']='NO'
    r['third_party_content_checked']='NO'
    r['commercial_use_confirmed']='NO'
    r['training_use_confirmed']='NO'
by_repo=collections.Counter(r.get('repo','UNKNOWN') for r in review)
by_topic=collections.Counter(r['suggested_topic_hint'] for r in review)
print('Review queue by repository:',dict(by_repo))
print('Review queue by topic hint:',dict(by_topic))

## 5. Export review artifacts

The review CSV preserves repository, commit, path, blob hash, and Stage 5 license-evidence paths. **No row is approved.**

In [ ]:
def export_csv(path,rows):
    if not rows: path.write_text('',encoding='utf-8'); return
    fields=sorted({k for r in rows for k in r})
    with path.open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore'); w.writeheader()
        for r in rows:
            w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in r.items()})
review_csv=OUTPUT/'stage6_human_review_queue.csv'
held_csv=OUTPUT/'stage6_held_candidates.csv'
summary_path=OUTPUT/'stage6_curation_summary.json'
export_csv(review_csv,review); export_csv(held_csv,held)
summary={
 'project':'Game Intelligence AI','stage':6,
 'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'input_file':AUDIT_PATH.name,'candidate_inventory_key':key,
 'input_candidate_count':len(candidates),'manual_review_queue_count':len(review),
 'held_or_excluded_count':len(held),'review_queue_by_repo':dict(by_repo),
 'review_queue_by_topic_hint':dict(by_topic),
 'source_files_downloaded':False,'training_dataset_created':False,
 'license_approval_granted':False,'training_authorized':False,
 'policy':'All review rows remain HUMAN_REVIEW_REQUIRED. No automated license approval.'
}
summary_path.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(summary,indent=2,ensure_ascii=False))

## 6. Download the reports

Run this cell to download the three review artifacts. Keep the original Stage 5 audit unchanged.

In [ ]:
from google.colab import files
for p in (review_csv,held_csv,summary_path): files.download(str(p))

## 7. Stop/go gate

**Do not start training after Stage 6.** The next stage should inspect exact file-level license notices and source headers for a deliberately small, human-selected set. Unclear, conflicting, third-party, or asset-related content stays excluded.

Success means the Stage 5 audit was parsed and triaged into auditable reports. No source content was extracted, approved, or used for training.